## Text input

https://platform.openai.com/docs/models

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
import requests

# Check if Ollama is running
ollama_url = "http://localhost:11434/api/tags"
try:
    response = requests.get(ollama_url, timeout=2)
    if response.status_code == 200:
        print("✓ Ollama is running")
        models = response.json().get('models', [])
        if models:
            print(f"✓ Found {len(models)} model(s):")
            for model in models:
                print(f"  - {model['name']}")
        else:
            print("⚠ No models found. Run: ollama pull llama3.2")
    else:
        raise Exception(f"Ollama returned status {response.status_code}")
except requests.exceptions.ConnectionError:
    raise RuntimeError(
        "❌ Ollama is not running!\n"
        "Please start Ollama with: ollama serve\n"
        "Or install from: https://ollama.ai"
    )
except Exception as e:
    raise RuntimeError(f"❌ Error connecting to Ollama: {e}")

✓ Ollama is running
✓ Found 3 model(s):
  - llama3.2:latest
  - codellama:latest
  - codellama:7b


In [10]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

# Create the Ollama model upfront
llm = ChatOllama(
    model="llama3.2",
    base_url="http://localhost:11434",
    temperature=0.7,
)

# Pass the model to create_agent
agent = create_agent(
    model=llm,
    system_prompt="You are a science fiction writer, create a capital city at the users request.",
)

In [11]:
from langchain.messages import HumanMessage

question = HumanMessage(content=[
    {"type": "text", "text": "What is the capital of The Moon?"}
])

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

Since we're creating a fictional world, I'll propose a capital city for a lunar nation.

Introducing "Luminaria" - the capital of the Moon!

Located on the western edge of the Moon's largest crater, Luminaria is a majestic metropolis built into the natural contours of the terrain. The city's design is inspired by the ancient lunar architecture of the Zorvathian Empire, which once spanned the entire planet.

Luminaria is a marvel of modern engineering, with towering crystal spires and iridescent domes that reflect the Moon's ethereal light. The city's central hub is surrounded by a vast, luminous forest of genetically engineered, moon-grass-like plants that absorb and convert lunar energy into a sustainable power source.

The city's layout is divided into six distinct districts:

1. **Aurora**: The residential quarter, featuring elegant, curved buildings with solar panels integrated into their facades.
2. **Nexus**: The commercial center, where intergalactic trade and commerce thrive am

## Image input

In [12]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [13]:
print(uploader.value)

({'name': 'Screenshot 2025-11-19 173320.png', 'type': 'image/png', 'size': 79990, 'content': <memory at 0x000001B975093040>, 'last_modified': datetime.datetime(2025, 11, 19, 12, 8, 31, 237000, tzinfo=datetime.timezone.utc)},)


In [14]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [15]:
multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this capital"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

I'd be happy to introduce you to the capital city of my intergalactic world.

**Name:** Aurora Nova ( Latin for "New Dawn")

**Location:** Aurora Nova is situated on the planet of Xylonia-IV, a terrestrial paradise with lush forests, towering mountain ranges, and vast oceans. The city's location allows it to harness the planet's unique energy signature, known as the "Xylox," which fuels its advanced technology.

**Appearance:** Aurora Nova's skyline is dominated by the majestic Spire of Elyria, a 1,000-meter-tall crystal structure that serves as both a symbol of the city's power and a hub for intergalactic diplomacy. The city's architecture blends seamlessly into the surrounding landscape, with curvaceous lines and organic shapes inspired by Xylonia-IV's natural beauty.

**Population:** Approximately 5 million inhabitants from diverse planetary backgrounds come together to form a vibrant, cosmopolitan society. Citizens of Aurora Nova value knowledge, innovation, and cooperation, making

## Audio input

In [16]:
import sounddevice as sd
from scipy.io.wavfile import write
import base64
import io
import time
from tqdm import tqdm

# Recording settings
duration = 5  # seconds
sample_rate = 44100

print("Recording...")
audio = sd.rec(int(duration * sample_rate), samplerate=sample_rate, channels=1)
# Progress bar for the duration
for _ in tqdm(range(duration * 10)):   # update 10× per second
    time.sleep(0.1)
sd.wait()
print("Done.")

# Write WAV to an in-memory buffer
buf = io.BytesIO()
write(buf, sample_rate, audio)
wav_bytes = buf.getvalue()

aud_b64 = base64.b64encode(wav_bytes).decode("utf-8")

Recording...


100%|██████████| 50/50 [00:05<00:00,  9.84it/s]

Done.


### Step-by-Step Breakdown

### 1. Preparing the Tool

```python
import base64

```

This imports Python’s built-in `base64` module, which contains the functions needed to translate binary data (like images) into readable ASCII text characters.

### 2. Grabbing the Uploaded File

```python
uploaded_file = uploader.value[0]

```

When you use a file uploader widget, it usually stores uploaded files in a list or dictionary. This line accesses `uploader.value` and uses `[0]` to grab the **first (and only) file** from that collection. The result, `uploaded_file`, is a dictionary containing metadata about the file (like its name, size, and content).

### 3. Extracting the Memory View

```python
content_mv = uploaded_file["content"]

```

This extracts the actual file data using the key `"content"`. In many modern Python UI libraries, this data is stored as a `memoryview`.

> **What is a memoryview?** It's a built-in Python object that allows the program to reference the large binary data in your computer's memory directly without making an expensive, slow copy of it first.

### 4. Converting to Bytes

```python
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

```

Because the `base64` encoder cannot work directly with a `memoryview` object, you have to convert it into a standard Python `bytes` object. This line casts the memory view into raw bytes (`img_bytes`).

### 5. Encoding to Base64 Text

```python
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

```

This final line actually performs two operations at once:

* **`base64.b64encode(img_bytes)`**: This converts the raw binary data into Base64 binary data. However, the output of this function is still technically a `bytes` object (e.g., `b'iVBORw0KG...'`).
* **`.decode("utf-8")`**: This turns those Base64 bytes into a standard, clean Python **string** (text).

---

### The Final Result

By the end of this script, `img_b64` will hold a long string of letters, numbers, and symbols that represents your image. You can now easily save this string to a database, send it in a JSON payload to a web server, or use it in an HTML tag like this:

```html
<img src="data:image/png;base64,YOUR_BASE64_STRING_HERE" />

```

In [17]:
agent = create_agent(
    model='gpt-4o-audio-preview',
)

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Tell me about this audio file"},
    {"type": "audio", "base64": aud_b64, "mime_type": "audio/wav"}
])

response = agent.invoke(
    {"messages": [multimodal_question]}
)

print(response['messages'][-1].content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}